In [1]:
import sys
import os
sys.path.append(os.path.abspath("../"))

In [2]:
from utils.loaders import load_sweep
from utils.styles import apply
from utils.analysis import cbs_profiles, linear, circular, keep, phi_cut, azimuthal_average

import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

apply(context="paper", col="single")

In [3]:
save_path = "/Users/niaggar/Results"

radius_values = [0.020, 0.035, 0.055, 0.075, 0.100, 0.175]
N_REPLICAS = 5

data = []
for rep in range(N_REPLICAS):
    for index, rad in enumerate(radius_values):
        name = f"radius_{rad:.3f}__rep{rep}"
        val = index * N_REPLICAS + rep
        full_name = f"{val:04d}_{name}"
        data.append((full_name, rf"R={rad*1000:.0f} $\mu$m"))

print(data)

[('0000_radius_0.020__rep0', 'R=20 $\\mu$m'), ('0005_radius_0.035__rep0', 'R=35 $\\mu$m'), ('0010_radius_0.055__rep0', 'R=55 $\\mu$m'), ('0015_radius_0.075__rep0', 'R=75 $\\mu$m'), ('0020_radius_0.100__rep0', 'R=100 $\\mu$m'), ('0025_radius_0.175__rep0', 'R=175 $\\mu$m'), ('0001_radius_0.020__rep1', 'R=20 $\\mu$m'), ('0006_radius_0.035__rep1', 'R=35 $\\mu$m'), ('0011_radius_0.055__rep1', 'R=55 $\\mu$m'), ('0016_radius_0.075__rep1', 'R=75 $\\mu$m'), ('0021_radius_0.100__rep1', 'R=100 $\\mu$m'), ('0026_radius_0.175__rep1', 'R=175 $\\mu$m'), ('0002_radius_0.020__rep2', 'R=20 $\\mu$m'), ('0007_radius_0.035__rep2', 'R=35 $\\mu$m'), ('0012_radius_0.055__rep2', 'R=55 $\\mu$m'), ('0017_radius_0.075__rep2', 'R=75 $\\mu$m'), ('0022_radius_0.100__rep2', 'R=100 $\\mu$m'), ('0027_radius_0.175__rep2', 'R=175 $\\mu$m'), ('0003_radius_0.020__rep3', 'R=20 $\\mu$m'), ('0008_radius_0.035__rep3', 'R=35 $\\mu$m'), ('0013_radius_0.055__rep3', 'R=55 $\\mu$m'), ('0018_radius_0.075__rep3', 'R=75 $\\mu$m'), ('0

# Linear Polarization study of RGD particles

In [7]:
def load_stitched(sweep, key, basis, reduce, channel, k, lstar, cbs_sensor_extra="", time_index=0):
    """Une det_1 (fino) y det_2 (cola) promediando la banda de solape.
    Devuelve theta [rad], q=k*lstar*theta, y el enhancement del canal."""
    p1 = cbs_profiles(sweep[key].processed_cbs(f"farfield_cbs_1{cbs_sensor_extra}"), basis=basis, time_index=time_index, reduce=reduce)
    p2 = cbs_profiles(sweep[key].processed_cbs(f"farfield_cbs_2{cbs_sensor_extra}"), basis=basis, time_index=time_index, reduce=reduce)
    th1, g1 = np.asarray(p1.theta), np.asarray(p1.enhancement[channel])
    th2, g2 = np.asarray(p2.theta), np.asarray(p2.enhancement[channel])

    theta_1 = th1[-1]                       # fin de la ventana fina
    lo = 0.9 * theta_1                       # inicio real de det_2 (solape)

    # --- chequeo de stitching: interpola det_1 sobre los bins de solape de det_2
    ov = (th2 >= lo) & (th2 <= theta_1)
    if ov.any():
        g1_on2 = np.interp(th2[ov], th1, g1)
        resid = np.abs(g2[ov] - g1_on2)
        if np.max(resid) > 0.05:            # umbral: 5% del enhancement
            print(f"[stitch WARN] {key} ch={channel}: max solape {np.max(resid):.3f} -- revisar normalizacion angulo solido")

    # une: det_1 completo + det_2 estrictamente por encima de theta_1
    keep2 = th2 > theta_1
    theta = np.concatenate([th1, th2[keep2]])
    g = np.concatenate([g1, g2[keep2]])
    q = k * lstar * theta
    return theta, q, g

In [4]:
folder = "study_rgd_events__PLIN__beam2500"
sweep_data = load_sweep(folder, base_path=Path(save_path))

data_keys = list(sweep_data.keys())
print(f"Loaded {len(data_keys)} datasets: {data_keys}")

Loaded 30 datasets: ['0012_radius_0.055__rep2', '0021_radius_0.100__rep1', '0001_radius_0.020__rep1', '0016_radius_0.075__rep1', '0025_radius_0.175__rep0', '0007_radius_0.035__rep2', '0009_radius_0.035__rep4', '0015_radius_0.075__rep0', '0026_radius_0.175__rep1', '0023_radius_0.100__rep3', '0010_radius_0.055__rep0', '0005_radius_0.035__rep0', '0027_radius_0.175__rep2', '0019_radius_0.075__rep4', '0003_radius_0.020__rep3', '0014_radius_0.055__rep4', '0029_radius_0.175__rep4', '0017_radius_0.075__rep2', '0006_radius_0.035__rep1', '0028_radius_0.175__rep3', '0002_radius_0.020__rep2', '0011_radius_0.055__rep1', '0022_radius_0.100__rep2', '0018_radius_0.075__rep3', '0004_radius_0.020__rep4', '0024_radius_0.100__rep4', '0008_radius_0.035__rep3', '0000_radius_0.020__rep0', '0020_radius_0.100__rep0', '0013_radius_0.055__rep3']


In [ ]:
# Flat repetitions


def mean_repetitions(standard_name, sweep, key, basis, reduce, channel, k, lstar, cbs_sensor_extra="", time_index=0):
    """Promedia las N_REPLICAS repeticiones de un mismo radio."""
    theta_list = []
    q_list = []
    g_list = []
    for rep in range(N_REPLICAS):
        name = f"{standard_name}__rep{rep}"
        val = key * N_REPLICAS + rep
        full_name = f"{val:04d}_{name}"
        theta, q, g = load_stitched(sweep, full_name, basis, reduce, channel, k, lstar,
                                    cbs_sensor_extra=cbs_sensor_extra,
                                    time_index=time_index)
        theta_list.append(theta)
        q_list.append(q)
        g_list.append(g)

    # Promedia los resultados
    theta_mean = np.mean(theta_list, axis=0)
    q_mean = np.mean(q_list, axis=0)
    g_mean = np.mean(g_list, axis=0)
    return theta_mean, q_mean, g_mean

